# 00-Setup: 10분 사전 점검

이 노트북은 수업 전에 끝내 두는 10분 점검입니다. 01~06을 시작하기 전에 내 컴퓨터에서 `rag_lab`, 한국어 토크나이저(Kiwi), 로컬 임베딩 모델, 재순위 모델, Deep Agents, 실제 모델 호출이 동작하는지 눈으로 확인합니다.

## 학습 목표

- 설치는 저장소 루트에서 한 번, 실행은 `Day02_new/`에서 한다는 구분을 말할 수 있다.
- 실제 모델을 부르려면 `Day02_new/.env`에 `ANTHROPIC_API_KEY`와 판정 API 키(`TYPESAFE_API_KEY`, `OPENROUTER_API_KEY`, `OPENAI_API_KEY` 중 하나)가 필요하고, 없으면 어디서 멈추는지 말할 수 있다.
- 내 환경에서 `rag_lab`, Kiwi, 로컬 임베딩 모델, 재순위 모델, Deep Agents, 실제 모델이 동작함을 확인한다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 설치와 폴더 구조 확인 | 2 |
| 2 | 키 상태 확인, `.env` 안내, 판정 API 한 번 | 3 |
| 3 | 토크나이저, 로컬 임베딩 모델, 재순위 모델 | 2 |
| 4 | 실제 모델 호출과 Deep Agents 한 턴 | 2 |
| 5 | 이후 시간표와 비용 규칙 | 1 |

## 호출 규모

이 노트북에서 외부 API를 부르는 셀은 셋입니다. 절 2의 판정 API 1회, 절 4의 직접 호출 1회와 Deep Agents 한 턴 1회로, 호출은 약 3회입니다(Deep Agents가 내부에서 더 부르면 늘 수 있습니다). 임베딩과 재순위는 내 컴퓨터의 로컬 모델이 처리하므로 API 호출이 아닙니다. 키가 없으면 절 2의 키 확인 셀과 판정 API 셀에서 한국어 안내와 함께 멈추므로 이후 호출까지 가지 않습니다.

## 1. 설치와 폴더 구조 (3분)

설치는 **저장소 루트에서 한 번만** 합니다. 루트 `pyproject.toml`이 uv workspace이고, `uv.lock`과 `.venv`는 루트에 하나만 있습니다.

```bash
uv sync --locked            # 저장소 루트에서 한 번만
cd Day02_new
uv run jupyter lab          # 실행은 Day02_new 폴더에서
```

`Day02_new/`에서 `uv sync`를 실행하지 않습니다. 이 폴더에 필요 없는 패키지가 환경에서 지워져 다른 Day 실습이 깨집니다. 잘못 실행했다면 루트에서 `uv sync --locked`로 복구합니다(루트 `AGENTS.md`의 실행 규칙).

아래 셀은 지금 이 kernel이 루트 `.venv`에서 돌고 있는지 확인합니다.

In [ ]:
import sys
from importlib.metadata import version

print("Python    :", sys.version.split()[0])
print("deepagents:", version("deepagents"))
print("sys.prefix:", sys.prefix)
print("루트 .venv에서 실행 중인가:", sys.prefix.endswith(".venv"))

In [ ]:
from rag_lab.settings import ROOT

skill_folders = sorted(p.name for p in (ROOT / "skills").iterdir() if (p / "SKILL.md").exists())
documents = sorted(p.name for p in (ROOT / "data" / "business").iterdir())

print("ROOT           :", ROOT.name)
print("skills/        :", skill_folders)
print("data/business/ :", len(documents), "개 파일")

`import rag_lab`이 되는 이유는 노트북이 `rag_lab/` 폴더 옆에 있고 kernel의 작업 폴더가 `Day02_new/`이기 때문입니다. 다른 폴더에서 kernel을 띄우면 이 import가 실패합니다. `## 직접 해보기` 2번에서 확인합니다.

## 2. 키 상태와 `.env` (2분)

키는 `Day02_new/.env`에만 둡니다. 직접 만들어야 합니다.

```bash
cp Day02_new/.env.example Day02_new/.env   # 그리고 편집기로 ANTHROPIC_API_KEY를 직접 채웁니다
```

- 키 값은 노트북, 로그, 결과 파일 어디에도 출력하지 않습니다. 이 수업의 도구는 `설정됨`/`미설정`만 말해 줍니다.
- 다른 Day의 `.env`나 셸에 export한 키는 읽지 않습니다. 모르는 사이에 내 크레딧이 쓰이는 사고를 막기 위해서입니다.
- 키를 `.env`에 넣은 뒤에는 **kernel을 재시작**해야 반영됩니다.
- 이 수업은 실제 모델만 씁니다. 키가 없으면 대체 경로로 넘어가지 않고 아래 셀에서 바로 멈춥니다.

In [ ]:
from rag_lab.settings import key_status, require_key

for name, value in key_status().items():
    print(f"{name:<24} {value}")

settings = require_key()  # 키가 없으면 여기서 한국어 안내와 함께 RuntimeError로 멈춥니다
print("사용 모델  :", settings.model)

### 판정 API 키 (필수)

이 수업은 `ANTHROPIC_API_KEY` 외에 **판정 API 키가 하나 더 필요합니다.** `TYPESAFE_API_KEY`, `OPENROUTER_API_KEY`, `OPENAI_API_KEY` 중 하나를 `Day02_new/.env`에 **직접** 채우세요. 대체 경로(skip)는 없어서 비어 있으면 아래 셀에서 멈춥니다. 여러 개를 채웠다면 `DAY02_DECISION_BACKEND`로 고르고, 비워 두면 위 순서대로 키가 있는 첫 번째를 씁니다.

- 04~06에서 질문 문장과 근거 텍스트가 선택한 공급자에게 전송됩니다. 외부에 보내면 안 되는 내용은 넣지 마세요.
- 채팅 모델은 문장을 자유롭게 생성하지만, 판정 API는 참/거짓 같은 정해진 질문에 확률을 숫자로 돌려주므로 문턱값(gate)을 코드로 걸 수 있습니다.

아래 셀은 해롭지 않은 참/거짓 질문 하나(`Noul`)로 `judge`를 한 번 부릅니다.

In [ ]:
import time
from rag_lab.settings import announce
from rag_lab.decisions import Noul, make_decider

announce("판정 API 확인", model_calls=1)
decider = make_decider()  # 키가 없으면 여기서 한국어 안내와 함께 RuntimeError로 멈춥니다
question = Noul("is_korean", "문장이 한국어로 쓰였는지 판정합니다.",
                true="문장이 한국어로 쓰였다.", false="문장이 한국어로 쓰이지 않았다.")

started = time.perf_counter()
judgment = decider.judge("서비스 크레딧은 다음 달 말일까지 신청합니다.", [question])
elapsed = time.perf_counter() - started

probability = judgment.verdicts["is_korean"].probability
print("backend    :", judgment.backend)
print("model      :", judgment.model)
print("probability:", probability)
print(f"걸린 시간  : {elapsed:.2f}초")
assert probability is not None and 0 <= probability <= 1, "확률은 0~1 사이여야 합니다"

## 3. 토크나이저, 로컬 임베딩 모델, 재순위 모델 (2분)

먼저 한국어 토크나이저가 동작하는지 봅니다. 첫 실행은 Kiwi가 사전을 읽느라 느립니다.

Anthropic에는 임베딩 API가 없으므로, 밀집 검색(dense retrieval)에 쓰는 임베딩은 내 컴퓨터에서 도는 로컬 모델이 만듭니다. 첫 사용 때 모델 파일을 내려받기 때문에 시간이 걸릴 수 있습니다.

In [ ]:
import time
from rag_lab.retrieval import tokenize

started = time.perf_counter()
tokens = tokenize("서비스 크레딧 신청 기한")
elapsed = time.perf_counter() - started

print("토큰:", tokens)
print(f"첫 실행 시간: {elapsed:.2f}초")

In [ ]:
from rag_lab.settings import make_embeddings

started = time.perf_counter()
vector = make_embeddings().embed_query("서비스 크레딧 신청 기한")
elapsed = time.perf_counter() - started

print("벡터 차원  :", len(vector))
print(f"적재 포함 시간: {elapsed:.2f}초")

**재순위 모델(reranker)은 필수입니다.** `BAAI/bge-reranker-v2-m3`는 04~06에서 후보를 다시 줄 세우는 데 쓰는 로컬 교차 인코더입니다. 첫 실행에 모델 파일 약 2.1GB를 내려받으므로, 연결이 좋은 곳에서 수업 전에 이 셀을 먼저 실행해 두세요. 로컬에서 돌기 때문에 API 호출도 비용도 없습니다.

In [ ]:
from rag_lab.settings import make_reranker

query = "서비스 크레딧 신청 기한"
relevant = "서비스 크레딧은 장애가 발생한 달의 다음 달 말일까지 신청해야 합니다."
irrelevant = "사내 식당은 평일 오전 11시 30분부터 오후 1시까지 운영합니다."

started = time.perf_counter()
scores = make_reranker().score(query, [relevant, irrelevant])
elapsed = time.perf_counter() - started

print(f"관련 문장 점수  : {scores[0]:.2f}")
print(f"무관한 문장 점수: {scores[1]:.2f}")
print(f"적재 포함 시간  : {elapsed:.2f}초")
assert scores[0] > scores[1], "관련 문장이 더 높아야 합니다"

### 방금 결과가 증명하는 것과 증명하지 않는 것

| 증명하는 것 | 증명하지 않는 것 |
|---|---|
| 패키지가 설치되어 import 된다 | 검색이 맞는 문서를 찾는다 |
| Kiwi가 한국어를 쪼갠다 | 임베딩 검색의 품질이 충분하다 |
| 로컬 임베딩 모델이 로드되어 벡터를 낸다 | 모델이 도구를 올바르게 고른다 |
| 재순위 모델이 로드되어 점수를 낸다 | 재순위가 검색 품질을 올린다 |

## 4. 실제 모델 호출과 Deep Agents 한 턴 (2분)

`announce(...)`가 실행 직전에 규모를 한 줄로 알립니다. 아래 두 셀은 실제 모델을 부르고, 이후 어떤 셀도 이 결과에 기대지 않습니다.

**관찰할 것**: 모델 응답 문장과 토큰 수는 실행마다 다릅니다. 노트북에는 측정값을 적어 두지 않으니 내 출력에서 직접 읽습니다. (1) `announce`가 규모 한 줄을 먼저 찍는지, (2) 응답이 `.text`로 문자열이 되어 나오는지, `내용 블록 종류`에 `thinking` 같은 블록이 섞이는지, (3) Deep Agents 한 턴의 `메시지 수`가 사용자 질문과 모델 응답을 합친 값인지 확인합니다.

In [ ]:
from rag_lab.settings import announce, make_model

announce("모델 호출 확인", model_calls=1)
reply = make_model(settings).invoke("한 단어로 답하세요: 준비")
print("모델 응답:", reply.text)
print("내용 블록 종류:", [b["type"] if isinstance(b, dict) else "text" for b in reply.content] if isinstance(reply.content, list) else "text")

In [ ]:
from deepagents import create_deep_agent

announce("Deep Agents 한 턴", model_calls=1)
agent = create_deep_agent(model=make_model(settings))
result = agent.invoke({"messages": [{"role": "user", "content": "한 단어로 답하세요: 준비"}]})

print("메시지 수  :", len(result["messages"]))
print("마지막 응답:", result["messages"][-1].text)

### 실패했을 때 볼 곳

| 증상 | 먼저 볼 곳 |
|---|---|
| 키 안내와 함께 `RuntimeError`가 난다 | `Day02_new/.env`가 있는지, 키를 채운 뒤 kernel을 재시작했는지 |
| 판정 API 셀에서 `RuntimeError` | `.env`에 세 판정 키 중 하나를 채웠는지, `DAY02_DECISION_BACKEND` 철자 |
| 401 오류 | 키 값 오타, 만료 여부(값을 출력하지 말고 해당 공급자 콘솔에서 확인) |
| 시간 초과 | 네트워크 연결(모델 호출은 120초 안에 끝나야 합니다) |
| `ModuleNotFoundError` | 루트에서 `uv sync --locked`를 했는지, kernel이 루트 `.venv`인지, 작업 폴더가 `Day02_new/`인지 |

## 5. 이후 시간표와 비용 규칙 (1분)

| 노트북 | 주제 | 시간 |
|---|---|---:|
| 01 | Agentic AI 기초 | 60분 |
| 02 | Function Calling과 Tools | 60분 |
| 03 | Agent Skills | 60분 |
| 04 | 검색 설계 | 60분 |
| 05 | 프로덕션 Agentic RAG | 60분 |
| 06 | DeepAgents RAG 실습 | 60분 |

비용 규칙: 실제 모델을 부르는 셀은 실행 직전에 규모를 한 줄로 알립니다(`이 실행은 모델 호출 약 N회...`). 키(`ANTHROPIC_API_KEY`와 판정 키)가 없으면 어떤 노트북도 실행되지 않고, 키가 있으면 비용은 내 크레딧에서 나갑니다.

## 직접 해보기

1. `.env` 없이 실행해 어디서 멈추는지 본 뒤, `.env`를 만들고 kernel을 재시작해서 `key_status()`가 어떻게 바뀌는지 봅니다.
2. `Day02_new/` 밖에서 kernel을 띄우고 `import rag_lab`이 어떻게 되는지 본 뒤 이유를 한 줄로 적습니다.
3. Deep Agents 셀의 질문을 바꿔 보고, 메시지 수와 마지막 응답이 어떻게 달라지는지 비교합니다.

## 확인 질문

1. 모델 호출 셀이 성공했는데도 검색이나 답변 품질에 대해 아무 말도 할 수 없는 이유는 무엇인가요?
2. 셸에 `ANTHROPIC_API_KEY`를 export해도 `key_status()`가 `미설정`이라고 나오는 이유는 무엇이고, 이 설계가 막으려는 사고는 무엇인가요?
3. `Day02_new/`에서 `uv sync`를 실행하면 다른 Day에 어떤 일이 생기나요?

## 정리

- 설치는 루트에서 `uv sync --locked` 한 번, 실행은 `Day02_new/`에서 합니다.
- 키는 `Day02_new/.env`에 직접 채우고, 출력은 `설정됨`/`미설정`뿐입니다.
- 이 수업은 실제 모델만 씁니다. 키가 없으면 `require_key()`에서 멈추고, 임베딩과 재순위는 로컬 모델이 처리합니다.
- 다음: `01` 노트북에서 Agent가 무엇인지 시작합니다.